<a href="https://colab.research.google.com/github/Anil-Pradhan-web/touch-grass-nature-advisor/blob/main/LazyCatchup_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
pip install -q gradio transformers accelerate torch

In [3]:
import torch
import gradio as gr
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load a free open-weight model for text-to-text generation (FLAN-T5 Large)
print("Loading open-weight model (google/flan-t5-large)... This might take a minute.")
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")
print("Model loaded successfully!")

def generate_response(prompt, max_len=128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_len)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def analyze_chat(raw_text):
    if not raw_text.strip():
        return "No text provided.", "Nothing to do, go back to sleep.", "Low"

    # Crafting the prompt for FLAN-T5 to perform the extraction structured tasks
    prompt_tldr = f"Summarize the following transcript or group chat in exactly two sentences. Be highly concise and direct.\n\nText:\n{raw_text}\n\nSummary:"
    prompt_actions = f"Identify actionable tasks or key actions specifically meant for the reader from the text below. If there are no clear action items, write 'Nothing to do, go back to sleep'.\n\nText:\n{raw_text}\n\nAction Items:"
    prompt_drama = f"Assess the level of urgency, conflict, or drama in the following text. Classify it strictly as either Low, Medium, or High, followed by a brief 1-sentence reason.\n\nText:\n{raw_text}\n\nUrgency & Drama Rating:"

    # Generate responses using explicit seq2seq generation
    tldr = generate_response(prompt_tldr, max_len=100)
    actions = generate_response(prompt_actions, max_len=100)
    drama = generate_response(prompt_drama, max_len=80)

    return tldr, actions, drama

# Build the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🦥 LazyCatchup AI")
    gr.Markdown("### The Anti-FOMO Group Chat & Meeting Digest (Hacktoberfest 'Slow and Steady' Edition)")
    gr.Markdown("Paste your messy unread group chats, long meeting transcripts, or overwhelming emails, and let our sloth-brained AI extract only what you absolutely *must* know while you stay relaxed.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                label="Raw, Messy or Long Text",
                placeholder="Paste group chats, slack messages, transcripts or emails here...",
                lines=10
            )
            submit_btn = gr.Button("☕ Catch Me Up (Slowly)", variant="primary")

        with gr.Column():
            output_tldr = gr.Textbox(label="🦥 Sloth TL;DR (2-Line Summary)", lines=3)
            output_actions = gr.Textbox(label="📋 Action Items (For the Lazy User)", lines=4)
            output_drama = gr.Textbox(label="🎭 Urgency & Drama Meter Rating", lines=2)

    # Examples for quick testing
    gr.Examples(
        examples=[
            ["Dave: Hey guys, we need to finalize the presentation by tomorrow morning!\nSarah: I still haven't received the figures from Mark.\nMark: I am working on them, stop stressing everyone out!\nDave: Just make sure it is done. @User please double-check the final slides before we present."],
            ["Alice: Hey, did anyone see the new policy update? It's pretty boring but we should probably read it.\nBob: Yeah, basically just says don't steal food from the office fridge.\nAlice: Cool, back to watching videos."]
        ],
        inputs=input_text
    )

    submit_btn.click(
        fn=analyze_chat,
        inputs=input_text,
        outputs=[output_tldr, output_actions, output_drama]
    )

# Launch the app with a public shareable link
demo.launch(share=True)

Loading open-weight model (google/flan-t5-large)... This might take a minute.


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!


/tmp/ipykernel_1123/2039754585.py:35: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d21913ea9030d132f5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
